# Progetto 5G Networks (Gruppo 2)
## Sorgente da trasmettere:

In [1]:
text = """A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspects of communication are irrelevant to the engineering problem. The significant aspect is that the actual message is one selected from a set of possible messages. The system must be designed to operate for each possible selection, not just the one which will actually be chosen since this is unknown at the time of design.   
If the number of messages in the set is finite then this number or any monotonic function of this number can be regarded as a measure of the information produced when one message is chosen from the set, all choices being equally likely. As was pointed out by Hartley the most natural choice is the logarithmic function. Although this definition must be generalized considerably when we consider the influence of the statistics of the message and when we have a continuous range of messages, we will in all cases use an essentially logarithmic measure. """
print(text)


A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in a general theory of communication. A basis for such a theory is contained in the important papers of Nyquist and Hartley on this subject. In the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original message and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system with certain physical or conceptual entities. These semantic aspects of communication are irrelev

## Codifica della sorgente

#### Algoritmi per la compressione LZ78

In [2]:
import math
import numpy as np

def lz78_compress_chars(text):
    dizionario = {"": 0}
    out = []
    w = ""
    for c in text:
        if w + c in dizionario:
            w += c
        else:
            out.append((dizionario[w], c))
            dizionario[w + c] = len(dizionario)
            w = ""
    if w:
        out.append((dizionario[w], None))
    return out


def lz78_decompress_chars(coppie):
    frasi = [""]
    out = []
    for idx, c in coppie:
        if c is None:
            out.append(frasi[idx])
        else:
            frase = frasi[idx] + c
            out.append(frase)
            frasi.append(frase)
    return "".join(out)

def _bit_char(alfabeto):
    # +1 per il valore riservato alla coppia finale senza simbolo (None)
    return max(1, math.ceil(math.log2(len(alfabeto) + 1)))


def lz78_to_bits(coppie, alfabeto):
    pos = {c: i for i, c in enumerate(alfabeto)}
    bc = _bit_char(alfabeto)
    out = []
    for k, (idx, c) in enumerate(coppie, start=1):
        n_idx = max(1, math.ceil(math.log2(k)))
        out.append(format(idx, f'0{n_idx}b'))
        val = pos[c] if c is not None else len(alfabeto)
        out.append(format(val, f'0{bc}b'))
    return np.array(list(''.join(out)), dtype=np.uint8)


def bits_to_lz78(bits, alfabeto):
    bc = _bit_char(alfabeto)
    s = ''.join(str(int(b)) for b in bits)
    coppie = []
    i = 0
    k = 1
    while i < len(s):
        n_idx = max(1, math.ceil(math.log2(k)))
        if i + n_idx + bc > len(s):
            raise ValueError("Flusso di bit troncato o corrotto")
        idx = int(s[i:i + n_idx], 2)
        i += n_idx
        val = int(s[i:i + bc], 2)
        i += bc

        if val == len(alfabeto):          # codice riservato: coppia finale
            c = None
        elif val < len(alfabeto):
            c = alfabeto[val]
        else:
            raise ValueError("Simbolo non valido nel flusso")

        if idx >= k:                      # indice oltre il dizionario corrente
            raise ValueError("Indice non valido nel flusso")

        coppie.append((idx, c))
        k += 1
    return coppie

#### Compressione della sorgente

In [3]:
compresso = lz78_compress_chars(text)
# print(compresso)
print(len(compresso), "coppie")

645 coppie


In [4]:
alfabeto = sorted(set(text))
bits = lz78_to_bits(compresso, alfabeto)
print(len(bits), "bits")

9298 bits


## Modulazione

In [41]:
import math
import numpy as np

# Parametri
N_bit = len(bits)
SNR_dB = 8

# Mappatura antipodale
simboli = 2 * bits.astype(int) - 1; 

# Rumore AWGN
SNR = 10 ** (SNR_dB / 10)          # conversione da dB a scala lineare
sigma = np.sqrt(1 / (2 * SNR))     # deviazione standard
# rumore = sigma * np.random.randn(N_bit)
rng = np.random.default_rng(1) # seed fisso per riproducibilità
rumore = sigma * rng.standard_normal(N_bit)

# Segnale ricevuto
ricevuti = simboli + rumore

bit_ricevuti = (ricevuti > 0).astype(int)

print(bit_ricevuti)
print(len(bit_ricevuti))

[0 0 0 ... 0 0 0]
9298


## Ricevitore

In [42]:
coppie_ricevute = bits_to_lz78(bit_ricevuti, alfabeto) # bit ricevuti dalla trasmissione
decompresso = lz78_decompress_chars(coppie_ricevute)
# print(decompresso)
print(len(coppie_ricevute), "coppie")

645 coppie


In [43]:
print(decompresso)

A Mathematical Theory of Communication
By C. E. SHANNON
INTRODUCTION
   THE recent development of various methods of modulation-such as PCM and PPM which exchange bandwidth for signal-to-noise ratio has intensified the interest in-a general theory of communication. A basis for such a theory is contained in-the important papers of Nyquist and Hartley on-this subject. In-the present paper we will extend the theory to include a number of new factors, in particular the effect of noise in the channel, and the savings possible due to the statistical structure of the original uessage and due to the nature of the final destination of the information.   
The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point. Frequently the messages have meaning; that is they refer to or are correlated according to some system withaiertain physical or conceptual entities. These semantic aspects of communication are irrelev

In [44]:
print(text == decompresso)

False


In [45]:
errori = [(i, a, b) for i, (a, b) in enumerate(zip(text, decompresso)) if a != b]
print(len(errori), "caratteri diversi")
for i, orig, ric in errori:
    print(f"pos {i}: atteso {orig!r}, ricevuto {ric!r}")

30 caratteri diversi
pos 127: atteso ' ', ricevuto '-'
pos 230: atteso ' ', ricevuto '-'
pos 307: atteso ' ', ricevuto '-'
pos 354: atteso ' ', ricevuto '-'
pos 371: atteso ' ', ricevuto '-'
pos 576: atteso 'm', ricevuto 'u'
pos 907: atteso ' ', ricevuto 'a'
pos 908: atteso 'c', ricevuto 'i'
pos 1074: atteso 'm', ricevuto 'u'
pos 1245: atteso ' ', ricevuto '-'
pos 1267: atteso ' ', ricevuto '-'
pos 1827: atteso 'g', ricevuto 's'
pos 1828: atteso 'a', ricevuto 't'
pos 1829: atteso 'r', ricevuto 'a'
pos 1830: atteso 'i', ricevuto 'r'
pos 1831: atteso 't', ricevuto 'i'
pos 1832: atteso 'h', ricevuto 't'
pos 1833: atteso 'm', ricevuto 'h'
pos 1834: atteso 'i', ricevuto 'm'
pos 1835: atteso 'c', ricevuto 'i'
pos 1836: atteso ' ', ricevuto 'c'
pos 1837: atteso 'm', ricevuto ' '
pos 1838: atteso 'e', ricevuto 'm'
pos 1839: atteso 'a', ricevuto 'e'
pos 1840: atteso 's', ricevuto 'a'
pos 1841: atteso 'u', ricevuto 's'
pos 1842: atteso 'r', ricevuto 'u'
pos 1843: atteso 'e', ricevuto 'r'
pos 184